# Part 2 - Restore on a new GPU machine and regenerate images

Use this on a fresh machine (no retraining). It:
1. recreates the cleaned Naruto split exactly as `p2_finetune.ipynb` did (seed 266),
2. fetches the two trained LoRA adapters (Hugging Face or a local copy),
3. regenerates the 20 prompts x 2 seeds for base, rank 4 and rank 16 with the original sampler settings.

Then run `p2_evaluate.ipynb`.

In [ ]:
# Cell 1 - run information (required)
import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})

In [ ]:
# Install only what is missing in the container (pinned to requirements.txt)
import importlib.util
need = {"diffusers": "diffusers==0.31.0", "transformers": "transformers==4.46.3", "accelerate": "accelerate==1.1.1",
        "peft": "peft==0.13.2", "datasets": "datasets==3.1.0", "open_clip": "open_clip_torch==2.29.0",
        "lpips": "lpips==0.1.4", "torchmetrics": "torchmetrics==1.6.0", "torch_fidelity": "torch-fidelity",
        "huggingface_hub": "huggingface_hub", "safetensors": "safetensors==0.4.5"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
print("installing:", missing)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

In [ ]:
import json, io, hashlib, time, shutil
from pathlib import Path
import pandas as pd
from PIL import Image

ROOT = Path.cwd()
for cand in [ROOT, ROOT.parent, Path("/app/DATA266-Lab2-Pair45"), Path("/workspace/lab2_package")]:
    if (cand / "part2_diffusion").exists():
        ROOT = cand; break
OUT = ROOT / "part2_diffusion"
MEMBER = OUT / "Nikhil_Kanaparthi"
CLEAN = OUT / "data" / "clean"; train_dir, test_dir = CLEAN / "train", CLEAN / "test"
BASE_MODEL = "runwayml/stable-diffusion-v1-5"
print({"root": str(ROOT)})

## 1. Recreate the cleaned split (identical procedure to p2_finetune cells 4-6)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("lambdalabs/naruto-blip-captions", split="train")
seen = set(); kept = []; removed_corrupt = 0; removed_duplicate = 0
for row in dataset:
    try:
        image = row["image"].convert("RGB")
        blob = io.BytesIO(); image.save(blob, format="PNG")
        digest = hashlib.sha256(blob.getvalue()).hexdigest()
        if digest in seen:
            removed_duplicate += 1; continue
        seen.add(digest)
        kept.append((image, str(row.get("text", "")).strip()))
    except Exception:
        removed_corrupt += 1
print({"loaded": len(dataset), "kept": len(kept), "removed_corrupt": removed_corrupt, "removed_duplicate": removed_duplicate})

rng = np.random.default_rng(SEED)
order = rng.permutation(len(kept)); split = int(0.9 * len(order))
train_ids, test_ids = order[:split], order[split:]
train_dir.mkdir(parents=True, exist_ok=True); test_dir.mkdir(parents=True, exist_ok=True)
def save_split(indices, folder):
    rows = []
    for j, idx in enumerate(indices):
        image, caption = kept[int(idx)]
        name = f"image_{j:05d}.png"
        image.resize((512, 512)).save(folder / name)
        rows.append({"file_name": name, "text": caption})
    (folder / "metadata.jsonl").write_text("\n".join(json.dumps(r) for r in rows))
    return rows
train_rows = save_split(train_ids, train_dir); test_rows = save_split(test_ids, test_dir)
print({"train": len(train_rows), "test": len(test_rows)})

prompts = json.loads((MEMBER / "prompts.json").read_text())
assert [p["prompt"] for p in prompts[:10]] == [r["text"] for r in test_rows[:10]], "split differs from the original run"
(OUT / "prompts.json").write_text(json.dumps(prompts, indent=2))
print("Split matches the original: the 10 in-domain prompts equal the first 10 test captions.")

## 2. Get the trained adapters

Option A (recommended): set `HF_USER` to where you uploaded them.
Option B: copy `pytorch_lora_weights.safetensors` into `part2_diffusion/lora_rank4/` and `part2_diffusion/lora_rank16/` yourself.

In [ ]:
HF_USER = "YOUR_HF_USERNAME"     # <- change if using option A
for m in ["rank4", "rank16"]:
    d = OUT / f"lora_{m}"; d.mkdir(parents=True, exist_ok=True)
    w = d / "pytorch_lora_weights.safetensors"
    if not w.exists() and HF_USER != "YOUR_HF_USERNAME":
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(f"{HF_USER}/data266-lab2-team45-naruto-lora-{m}", "pytorch_lora_weights.safetensors")
        shutil.copy(p, w)
    shutil.copy(MEMBER / f"lora_{m}" / "run_metrics.json", d / "run_metrics.json")   # original training metrics
    print(m, "adapter present:", w.exists(), w.stat().st_size if w.exists() else None)

## 3. Regenerate 40 images per model (same sampler: 30 steps, guidance 7.5, seeds 0 and 1)

In [ ]:
from diffusers import StableDiffusionPipeline
def generate_images(adapter_dir, name):
    pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL, torch_dtype=torch.float16).to("cuda")
    pipe.set_progress_bar_config(disable=True)
    if adapter_dir:
        pipe.load_lora_weights(str(adapter_dir))
    out_dir = OUT / "generations" / name; out_dir.mkdir(parents=True, exist_ok=True)
    rows, t0 = [], time.time()
    for item in prompts:
        for seed in (0, 1):
            g = torch.Generator(device="cuda").manual_seed(seed)
            res = pipe(item["prompt"], num_inference_steps=30, guidance_scale=7.5, generator=g)
            path = out_dir / f"prompt_{item['prompt_id']:02d}_seed_{seed}.png"
            res.images[0].save(path)
            rows.append({"model": name, "prompt_id": item["prompt_id"], "prompt": item["prompt"], "seed": seed,
                         "nsfw_flagged": bool(res.nsfw_content_detected[0]) if res.nsfw_content_detected else False,
                         "path": str(path.relative_to(ROOT))})
    pd.DataFrame(rows).to_csv(out_dir / "metadata.csv", index=False)
    secs = time.time() - t0
    print({"model": name, "images": len(rows), "nsfw_flagged": int(sum(r["nsfw_flagged"] for r in rows)),
           "seconds": round(secs, 1), "sec_per_image": round(secs / len(rows), 2)})
    del pipe; torch.cuda.empty_cache()

generate_images(None, "base")
generate_images(OUT / "lora_rank4", "rank4")
generate_images(OUT / "lora_rank16", "rank16")